<a href="https://colab.research.google.com/github/miray7yuce/quadcopter-rl-copilot/blob/main/quadcopter_rl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install stable-baselines3 gymnasium --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 2.4 MB/s eta 0:00:00


In [2]:
!pip uninstall jsbsim -y --quiet
!pip install jsbsim==1.2.4 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 15.6 MB/s eta 0:00:00


In [3]:
import jsbsim
print(jsbsim.__version__)

1.2.4


In [5]:
import jsbsim
import numpy as np
import gymnasium as gym
from gymnasium import spaces

class JSBSimQuadcopterEnv(gym.Env):
    def __init__(self):
        super().__init__()

        # Action: 4 motorun throttle komutu, 0-1 arası
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(4,), dtype=np.float32
        )

        # Observation: [altitude, roll, pitch, yaw, roll_rate, pitch_rate, yaw_rate, vc_fps]
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(8,), dtype=np.float32
        )

        self.fdm = None
        self.target_altitude = 10  # ft
        self.max_steps = 1000
        self.current_step = 0

        self.physics_dt = 1/1000
        self.control_dt = 1/50
        self.substeps = int(self.control_dt / self.physics_dt)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.fdm = jsbsim.FGFDMExec(None)
        self.fdm.load_model('F450')
        self.fdm.set_dt(self.physics_dt)

        # SCAS'ı kapalı, ham motor kontrolü
        self.fdm.set_property_value('fcs/ScasEngage', 0)

        self.fdm.run_ic()
        self.current_step = 0

        obs = self._get_obs()
        return obs, {}

    def _get_obs(self):
        altitude = self.fdm.get_property_value('position/h-sl-ft')
        roll = self.fdm.get_property_value('attitude/roll-rad')
        pitch = self.fdm.get_property_value('attitude/pitch-rad')
        yaw = self.fdm.get_property_value('attitude/heading-true-rad')
        roll_rate = self.fdm.get_property_value('velocities/p-rad_sec')
        pitch_rate = self.fdm.get_property_value('velocities/q-rad_sec')
        yaw_rate = self.fdm.get_property_value('velocities/r-rad_sec')
        vc = self.fdm.get_property_value('velocities/vc-fps')
        return np.array([altitude, roll, pitch, yaw, roll_rate, pitch_rate, yaw_rate, vc], dtype=np.float32)

    def step(self, action):
        action = np.clip(action, -1.0, 1.0)
        action = (action + 1.0)/2.0


        # 4 motora ayrı komut
        self.fdm.set_property_value('fcs/throttle-cmd-norm', float(action[0]))
        self.fdm.set_property_value('fcs/throttle-cmd-norm[1]', float(action[1]))
        self.fdm.set_property_value('fcs/throttle-cmd-norm[2]', float(action[2]))
        self.fdm.set_property_value('fcs/throttle-cmd-norm[3]', float(action[3]))


        for _ in range (self.substeps):
            self.fdm.run()

        self.current_step += 1

        obs = self._get_obs()
        altitude, roll, pitch = obs[0], obs[1], obs[2]

        altitude_error = abs(altitude - self.target_altitude)
        tilt_penalty = abs(roll) + abs(pitch)
        reward = float(-altitude_error - 0.1 * tilt_penalty)

        terminated = bool(altitude < 0 or altitude > 200 or abs(roll) > 1.5 or abs(pitch) > 1.5)
        truncated = bool(self.current_step >= self.max_steps)

        return obs, reward, terminated, truncated, {}

In [7]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

env = JSBSimQuadcopterEnv()
check_env(env, warn=True)
print("Ortam kontrolü tamam.")

model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    device="cpu"
)

model.learn(total_timesteps=100_000)
model.save("ppo_jsbsim_quadcopter")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Ortam kontrolü tamam.
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 64.2     |
|    ep_rew_mean     | -300     |
| time/              |          |
|    fps             | 332      |
|    iterations      | 1        |
|    time_elapsed    | 6        |
|    total_timesteps | 2048     |
---------------------------------


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 67.5         |
|    ep_rew_mean          | -347         |
| time/                   |              |
|    fps                  | 349          |
|    iterations           | 2            |
|    time_elapsed         | 11           |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0046038358 |
|    clip_fraction        | 0.0381       |
|    clip_range           | 0.2          |
|    entropy_loss         | -5.68        |
|    explained_variance   | -0.00106     |
|    learning_rate        | 0.0003       |
|    loss                 | 1.2e+03      |
|    n_updates            | 10           |
|    policy_gradient_loss | -0.00286     |
|    std                  | 1            |
|    value_loss           | 2.88e+03     |
------------------------------------------
------------------------------------------
| rollout/ 

In [9]:
import jsbsim

fdm = jsbsim.FGFDMExec(None)
print("kok dizin :", jsbsim.get_default_root_dir())
print("F450 yuklendi mi :", fdm.load_model('F450'))
print("varsayilan dt :", fdm.get_delta_t())

kok dizin : /usr/local/lib/python3.13/dist-packages/jsbsim
F450 yuklendi mi : True
varsayilan dt : 0.008333333333333333


In [10]:
fdm.set_dt(1/240)

fdm['ic/h-agl-ft']   = 3
fdm['ic/u-fps']      = 0
fdm['ic/v-fps']      = 0
fdm['ic/w-fps']      = 0
fdm['ic/phi-rad']    = 0
fdm['ic/theta-rad']  = 0
fdm.run_ic()

for i in range(4):
    fdm[f'propulsion/engine[{i}]/set-running'] = 1
fdm['fcs/ScasEngage'] = 0

for k in range(480):
    for i in range(4):
        fdm[f'fcs/throttle-cmd-norm[{i}]'] = 0.7
    fdm.run()
    if k % 120 == 0:
        print(
            f"t={fdm.get_sim_time():5.2f}  "
            f"h={fdm['position/h-agl-ft']:7.2f}  "
            f"hdot={fdm['velocities/h-dot-fps']:7.2f}  "
            f"thr0={fdm['fcs/throttle-cmd-norm[0]']:.3f}  "
            f"rpm0={fdm['propulsion/engine[0]/propeller-rpm']:8.1f}  "
            f"itki0={fdm['propulsion/engine[0]/thrust-lbs']:6.2f}"
        )

t= 0.00  h=   3.00  hdot=  -0.13  thr0=0.700  rpm0=     0.0  itki0=  0.00
t= 0.50  h=   5.91  hdot=  16.69  thr0=0.700  rpm0=  8030.5  itki0=  1.83
t= 1.00  h=  19.07  hdot=  34.59  thr0=0.700  rpm0=  8207.6  itki0=  1.49
t= 1.50  h=  39.26  hdot=  45.15  thr0=0.700  rpm0=  8340.9  itki0=  1.25


In [11]:
import jsbsim

def dene(thr, sure=3.0, h0=50.0):
    fdm = jsbsim.FGFDMExec(None)
    fdm.load_model('F450')
    fdm.set_dt(1/240)
    fdm['ic/h-agl-ft'] = h0
    for p in ['ic/u-fps', 'ic/v-fps', 'ic/w-fps', 'ic/phi-rad', 'ic/theta-rad']:
        fdm[p] = 0
    fdm.run_ic()
    for i in range(4):
        fdm[f'propulsion/engine[{i}]/set-running'] = 1
    fdm['fcs/ScasEngage'] = 0
    for _ in range(int(sure * 240)):
        for i in range(4):
            fdm[f'fcs/throttle-cmd-norm[{i}]'] = thr
        fdm.run()
    return fdm['position/h-agl-ft'] - h0, fdm['velocities/h-dot-fps'], fdm['inertia/weight-lbs']

for thr in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    dh, hdot, w = dene(thr)
    print(f"thr={thr:.2f}  3sn sonra dh={dh:+7.2f} ft   hdot={hdot:+7.2f} fps   agirlik={w:.2f} lbs")

thr=0.30  3sn sonra dh= -49.43 ft   hdot=  -7.58 fps   agirlik=3.09 lbs
thr=0.35  3sn sonra dh= -40.61 ft   hdot= -23.95 fps   agirlik=3.09 lbs
thr=0.40  3sn sonra dh= -13.38 ft   hdot=  -5.90 fps   agirlik=3.09 lbs
thr=0.45  3sn sonra dh= +14.29 ft   hdot= +11.33 fps   agirlik=3.09 lbs
thr=0.50  3sn sonra dh= +37.53 ft   hdot= +23.08 fps   agirlik=3.09 lbs
thr=0.55  3sn sonra dh= +58.26 ft   hdot= +32.42 fps   agirlik=3.09 lbs
thr=0.60  3sn sonra dh= +77.61 ft   hdot= +40.72 fps   agirlik=3.09 lbs
